# Tutorial 8A — Custom Faster R CNN and real annotations

Build a small CNN backbone, attach an RPN and RoI head, train on public pedestrian annotations, and calculate IoU and AP. The framework supplies tested anchor matching, losses and NMS instead of an incomplete detector mock-up.

**How to run:** select a Python kernel with the repository requirements, then Run All. QUICK mode keeps experiments short. For the larger run set `TUTORIAL_FULL=1` before starting the kernel. A quick run demonstrates the method; it is not a final accuracy benchmark.

**Execution status:** Complete implementation; end-to-end training/inference has not been verified in this repository run because the required large downloads did not finish. See `VALIDATION.md` for the separate offline model checks. Run All in Colab to generate your own results.

In [ ]:
# Google Colab: fetch the support files and install missing dependencies.
# Local Jupyter: follow README setup once; this cell makes no changes.
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    repo = Path("/content/dl-course-tutorials")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "https://github.com/naveed952/Tutorial-01_Perceptron_py.ipynb.git", str(repo)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo/"requirements.txt")])
    import os
    os.chdir(repo)


## Step 1 — Locate the project and set a reproducible seed
A fixed seed makes comparisons easier to reproduce. Each notebook can run independently.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd() if (Path.cwd()/"tutorial_support").exists() else Path.cwd().parent
assert (ROOT/"tutorial_support").exists(), "Run from the repository or notebooks folder."
sys.path.insert(0, str(ROOT))
from tutorial_support.common import *
seed()
print("Mode:", "QUICK: small real-data run" if QUICK else "FULL: larger training run")

## Step 2 — Load public labelled images and inspect the split
Penn–Fudan contains 170 images. Instance masks give bounding boxes directly; no manual annotations are invented. Full mode uses 150 train, 10 validation and 10 test images. Quick mode uses smaller disjoint subsets. Images from the same street scene can be correlated, so this small benchmark is only a learning exercise.

In [ ]:
import torch
from torch import nn
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign,box_iou,roi_pool
from tutorial_support.detection import *
DEVICE=torch_setup();dataset=PennFudan();train,val,test=loaders(dataset)
print('Dataset:',len(dataset),'split:',[len(x.dataset) for x in (train,val,test)])
image,target=dataset[0];show_prediction(image,{**target,'scores':torch.ones(len(target['boxes']))},0)

## Step 3 — Build backbone RPN and RoI heads
The RPN predicts foreground/background plus anchor offsets. RoI pooling or RoIAlign makes proposals a fixed size; the final head predicts person/background and refined boxes. Torchvision uses RoIAlign, a more precise alternative to quantized RoI pooling.

In [ ]:
def make_detector(channels=64):
    backbone=nn.Sequential(nn.Conv2d(3,32,3,stride=2,padding=1),nn.ReLU(),nn.Conv2d(32,channels,3,stride=2,padding=1),nn.ReLU())
    backbone.out_channels=channels
    anchors=AnchorGenerator(sizes=((32,64,128,256),),aspect_ratios=((.5,1.,2.),))
    pool=MultiScaleRoIAlign(featmap_names=['0'],output_size=7,sampling_ratio=2)
    return FasterRCNN(backbone,num_classes=2,rpn_anchor_generator=anchors,box_roi_pool=pool,min_size=128 if QUICK else 320,max_size=192 if QUICK else 480,rpn_pre_nms_top_n_train=200,rpn_post_nms_top_n_train=100,rpn_pre_nms_top_n_test=100,rpn_post_nms_top_n_test=50)
model=make_detector();print('Parameters:',sum(p.numel() for p in model.parameters()))
feature=torch.arange(64,dtype=torch.float32).reshape(1,1,8,8)
print('Classic RoI pooling example:',roi_pool(feature,[torch.tensor([[0.,0.,7.,7.]])],(2,2)))

## Step 4 — Train and compare two backbone widths
The loss dictionary contains objectness, RPN box regression, class prediction and final box regression. Box losses use matched positive examples. We select the width using validation AP50, then evaluate only that model on test.

In [ ]:
import copy,pandas as pd
rows=[];best=None;best_ap=-1
for width in [32,64]:
    torch.manual_seed(SEED);model=make_detector(width)
    history=train_detector(model,train,DEVICE,1 if QUICK else 10)
    metrics=evaluate_detector(model,val,DEVICE)
    rows.append({'width':width,**metrics})
    if metrics['map_50']>best_ap:best_ap=metrics['map_50'];best=copy.deepcopy(model).cpu();selected=width
    del model
display(pd.DataFrame(rows));final=evaluate_detector(best,test,DEVICE)
print('Selected width:',selected,'test metrics:',final)
save_result('08A_custom_detector',{'validation_comparison':rows,'selected_width':selected,'test':final})
image,target=test.dataset[0];best.eval()
with torch.inference_mode():prediction=best([image.to(DEVICE)])[0]
show_prediction(image,prediction,.2)

## Step 5 — Check IoU and box encoding numerically
The PDF’s second-stage centre-offset example omits multiplying by proposal width/height. We use the consistent standard: x = xa + tx·wa and w = wa·exp(tw). Smooth L1 acts on prediction minus target, not the target offsets alone.

In [ ]:
a=torch.tensor([[0.,0.,10.,10.]])
b=torch.tensor([[5.,5.,15.,15.]])
print('IoU:',box_iou(a,b).item());assert abs(box_iou(a,b).item()-25/175)<1e-6
anchor=np.array([80.,80.,120.,120.]);gt=np.array([100.,100.,100.,100.])
t=np.r_[(gt[:2]-anchor[:2])/anchor[2:],np.log(gt[2:]/anchor[2:])]
decoded=np.r_[anchor[:2]+t[:2]*anchor[2:],anchor[2:]*np.exp(t[2:])]
print('Offsets:',t,'decoded:',decoded);assert np.allclose(decoded,gt)

## Step 6 — Interpret precision recall and mAP
Sort detections by confidence. Match each prediction to at most one ground truth of the same class at the chosen IoU; unmatched predictions are false positives and missed annotations are false negatives. Precision = TP/(TP+FP); recall = TP/(TP+FN). AP summarizes the precision–recall curve. Report AP50 separately from COCO mAP averaged over IoU 0.50:0.95. Our evaluator uses TorchMetrics/COCO matching, not an arbitrary mean of precision values. A one-epoch model may have AP near zero; that is a result, not successful convergence.

[Dataset and detection reference](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html).

## Explain it in class
Describe the input, target, model, loss, update rule and evaluation split. Use the displayed results from your own run. Changing the dataset, seed or training budget can change the result.

**Course source:** the matching tutorial PDF supplied in the Deep Learning course. Original explanatory implementation; the course PDFs are not redistributed.